# 01 — Plant Flowering Data (Full Scale)

Loads all 6,466 plant species from PhenoField (107 parquet shards).
Builds the full plant flowering dataset used in the ANTHEIA pipeline.

**Output:** `stage4_F_existence_phenofield.csv` — binary plant existence matrix F (6,466 species × 3,162 CONUS bins)

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path
import gc

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path("/scratch/ariana.l")
PPE_DIR     = BASE / "ppe-outputs" / "opportunity_surface"
OUT_DIR     = BASE / "Stage 4 Link Prediction Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

F_OUT       = OUT_DIR / "stage4_F_existence_phenofield.csv"

# ── Constants ───────────────────────────────────────────────────────────────
BIN_SIZE    = 0.5

# CONUS bounding box
LAT_MIN, LAT_MAX = 24.0, 49.5
LON_MIN, LON_MAX = -125.0, -66.0

print("Paths OK")
print(f"  PPE_DIR : {PPE_DIR}")
print(f"  F_OUT   : {F_OUT}")

In [ ]:
# Stream all 107 PhenoField parquet shards
# Build binary plant existence matrix: F[species, bin] = 1 if observed
print("Streaming PhenoField parquet files...")

dataset = ds.dataset(str(PPE_DIR), format="parquet")

df = dataset.to_table(
    columns=["species", "lat", "lon"]
).to_pandas()

print(f"  Total records: {len(df):,}")
print(f"  Total species: {df['species'].nunique():,}")

In [ ]:
# Filter to CONUS bounding box
df = df[
    (df["lat"] >= LAT_MIN) & (df["lat"] <= LAT_MAX) &
    (df["lon"] >= LON_MIN) & (df["lon"] <= LON_MAX)
].copy()

print(f"  After CONUS filter: {len(df):,} records, {df['species'].nunique():,} species")

In [ ]:
# Add spatial bins
# Bin format: "lat_lon", e.g. "34.5_-120.0"
df["lat_bin"] = (np.floor(df["lat"] / BIN_SIZE) * BIN_SIZE).round(1)
df["lon_bin"] = (np.floor(df["lon"] / BIN_SIZE) * BIN_SIZE).round(1)
df["bin"]     = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

print(f"  Unique bins: {df['bin'].nunique():,}")

In [ ]:
# Build binary existence matrix F
print("Building F existence matrix...")

# Get all unique bins (sorted for reproducibility)
all_bins = sorted(df["bin"].unique())
all_species = sorted(df["species"].unique())

print(f"  Species: {len(all_species):,}")
print(f"  Bins:    {len(all_bins):,}")

# Build presence dict
presence = df.groupby("species")["bin"].apply(set).to_dict()

# Convert to binary DataFrame
rows = []
for sp in all_species:
    row = {b: (1 if b in presence[sp] else 0) for b in all_bins}
    rows.append(row)

F_df = pd.DataFrame(rows, index=all_species, columns=all_bins)

sparsity = 1 - F_df.values.mean()
print(f"  F matrix shape : {F_df.shape}")
print(f"  Sparsity       : {sparsity:.4f}")

del rows, presence
gc.collect()

In [ ]:
# Save F matrix
F_df.to_csv(F_OUT)
print(f"Saved → {F_OUT}")
print(f"Shape: {F_df.shape}")
F_df.iloc[:5, :5]